<a href="https://colab.research.google.com/github/timothy-shiwprasad/COMP6905_assigment_2/blob/main/Assignment_2_Question_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question 1 : Deliveries to Pokemon Center.

## Estimated Delivery Time (hours)

| Pokemon   | Pewter | Cerulean | Vermilion | Celadon | Fuchsia |
|-----------|--------|----------|------------|---------|---------|
| Pidgeot   | 7      | 5        | 6          | 2       | 9       |
| Dragonite | 5      | 4        | 7          | 3       | 8       |
| Lapras    | —      | 9        | 4          | 10      | 5       |
| Rapidash  | 6      | 7        | 9          | 4       | 8       |
| Arcanine  | 8      | 6        | 10         | 5       | 7       |


## A)

Goal: Minimize the Delivery time of the 5 Pokemon to their destinations.

This is an assigment problem. As every Center receives 1 package and each Pokemon can carry 1 package or 1 Pokemon to 1 Center.

## Sets :

### Pokemon (Supply):

P = {Pidgeot,Dragonite,Lapras,Rapidash,Arcanine}

### Pokemon Centers (Demand)

C = {Pewter ,Cerulean,Vermillion,Celadon,Fuchsia}

## Decison Variable:

$$
x_{ij} =
\begin{cases}
1, & \text{if Pokémon } i \text{ delivers to Center } j \\
0, & \text{otherwise}
\end{cases}
$$

$i \in P,\; j \in C$

## Cost Function
$c_{ij}$  = Cost associated with Pokemon i taking the package to Center j.

## Constraints

### Supply Constraint

Each Pokemon i,

$$
\sum_{j} x_{ij} = 1
$$

Each  Pokemon makes exactly one Delivery

### Demand Constraint

Each Center will only receive one package exactly

$$
\sum_{i} x_{ij} = 1
$$


## Final Formulation

$$ min \sum_{i} \sum_{j} c_{ij}.x{ij}$$

Subject to:

$$
\sum_{j} x_{ij} = 1
$$

$$
\sum_{i} x_{ij} = 1
$$

$$
x_{ij} ∈ {0,1}
$$


Note:
 Since the Pokemon Lapras cannot take the route to Pewter we will set to 0  in our Arc set. (Prohibited Route)

  

In [ ]:
# Question 1 B

import numpy as np
from scipy.optimize import linprog

def solve_assignment(cost,P,C):
    n = cost.shape[0]
    arcs = [(i,j) for i in range(n) for j in range(n) if not np.isnan(cost[i,j])]
    c = [cost[i,j] for i,j in arcs]
    A_eq, b_eq = [], []
    for i in range(n):                       # each Pokémon delivers once
        A_eq.append([1 if a==i else 0 for a,b in arcs]); b_eq.append(1)
    for j in range(n):                       # each Center receives once
        A_eq.append([1 if b==j else 0 for a,b in arcs]); b_eq.append(1)
    res = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=(0,None), method="highs")
    chosen = [(P[i],C[j]) for (i,j),v in zip(arcs,res.x) if v > 0.5]
    return res.fun, chosen, res.x

In [ ]:
P = ["Pidgeot","Dragonite","Lapras","Rapidash","Arcanine"]
C = ["Pewter","Cerulean","Vermilion","Celadon","Fuchsia"]
T = np.array([[7,5,6,2,9],
              [5,4,7,3,8],
              [np.nan,9,4,10,5],
              [6,7,9,4,8],
              [8,6,10,5,7]])

optimal,assigments,solution = solve_assignment(T,P,C)

print(f"The Minimum Delivery Time is {optimal}")

for assigment in assigments:
    print(f"{assigment[0]} is assigned to deliver the package to {assigment[1]}")

The Minimum Delivery Time is 23.0
Pidgeot is assigned to deliver the package to Celadon
Dragonite is assigned to deliver the package to Cerulean
Lapras is assigned to deliver the package to Vermilion
Rapidash is assigned to deliver the package to Pewter
Arcanine is assigned to deliver the package to Fuchsia


In [ ]:
# Question 1 C

# The Pidgeot-Celadon route is unavaliable so we will update our Cost Matrix
# T[0,3] = np.nan

# Cost
T_C = np.array([[7,5,6,np.nan,9],
              [5,4,7,3,8],
              [np.nan,9,4,10,5],
              [6,7,9,4,8],
              [8,6,10,5,7]])

optimal,assigments,solutions = solve_assignment(T_C,P,C)

print(f"The Minimum Delivery Time is {optimal}")

for assigment in assigments:
    print(f"{assigment[0]} is assigned to deliver the package to {assigment[1]}")

The Minimum Delivery Time is 25.0
Pidgeot is assigned to deliver the package to Cerulean
Dragonite is assigned to deliver the package to Pewter
Lapras is assigned to deliver the package to Vermilion
Rapidash is assigned to deliver the package to Celadon
Arcanine is assigned to deliver the package to Fuchsia


The New Minimum is 25.0
One Optimal assigment is is Lapras to Vermilion

The Minimum has increased by 2

## Question 1 D

The Cost value has now changed with the introduction of a deadline penalty. We will calclate the new cost as follows

New Cost  =    Delivery Time +  Late Penalty * max(0,Delivery Time - Deadline)


Deadline and Lateness penalty are all mesasured in time-equivalent units which allows all variables to be computed together.

In [ ]:
# Calculating New Cost

d   = np.array([5, 5, 4, 4, 5])      # deadlines
pen = np.array([3, 2, 4, 2, 6])      # penalty per hour late

late   = np.maximum(0, T - d)        # NaN stays NaN
C_pen  = T + pen * late              # combined cost matrix

print(C_pen)

## Solving the Model

optimal,assigments,solutions = solve_assignment(C_pen,P,C)



[[13.  5. 14.  2. 33.]
 [ 5.  4. 19.  3. 26.]
 [nan 17.  4. 22.  5.]
 [ 9. 11. 29.  4. 26.]
 [17.  8. 34.  7. 19.]]


In [ ]:
# Break the objective into its two parts
arcs = [(i, j) for i in range(5) for j in range(5) if not np.isnan(C_pen[i, j])]
sel  = [(i, j) for (i, j), v in zip(arcs, solutions) if v > 0.5]
total_time    = sum(T[i, j] for i, j in sel)
total_penalty = sum(pen[j] * late[i, j] for i, j in sel)

print(optimal, total_time, total_penalty)
print(f"The Combined Objective Value: {optimal}")
print(f"Total Delivery Time: {total_time}")
print(f"Total Penalty: {total_penalty}")

for i, j in sel:
    print("Assigment")
    print(P[i], "->", C[j])
    print(f"Time Taken: {T[i,j]}")
    print(f"Hours late: {late[i,j]}")
    print(f"Late Penalty: {late[i,j] * pen[j]}")

36.0 26.0 10.0
The Combined Objective Value: 36.0
Total Delivery Time: 26.0
Total Penalty: 10.0
Assigment
Pidgeot -> Vermilion
Time Taken: 6.0
Hours late: 2.0
Late Penalty: 8.0
Assigment
Dragonite -> Pewter
Time Taken: 5.0
Hours late: 0.0
Late Penalty: 0.0
Assigment
Lapras -> Fuchsia
Time Taken: 5.0
Hours late: 0.0
Late Penalty: 0.0
Assigment
Rapidash -> Celadon
Time Taken: 4.0
Hours late: 0.0
Late Penalty: 0.0
Assigment
Arcanine -> Cerulean
Time Taken: 6.0
Hours late: 1.0
Late Penalty: 2.0


# Question 1.d

The Combined objective Value: 36
The Total Delivery Time: 26
The total Penalty: 10


| Pokémon | Center | Time Taken (hours) | Hours Late | Late Penalty |
|---|---|---:|---:|---:|
| Pidgeot | Vermilion | 6.0 | 2.0 | 8.0 |
| Dragonite | Pewter | 5.0 | 0.0 | 0.0 |
| Lapras | Fuchsia | 5.0 | 0.0 | 0.0 |
| Rapidash | Celadon | 4.0 | 0.0 | 0.0 |
| Arcanine | Cerulean | 6.0 | 1.0 | 2.0 |

# Question 1 D

## Comparison with solution from part b.

In part B the minimum is 23. There are 2 assigments which would be late , Rapidash to Pewter and Arcanine to Fuchsia. They are 1 and 2 hours late and with late penalities 3 and 12 respectively. The late penalty and minimum value together would result in a combined minimum of 38. This is worse than our recently computed minimum of 36. The additional 3 hours of delivery time has given a 5 unit reduction in lateness penalty.

## Why integrality constraints are unnecessary

Integrality constraints are unnecessary because the assignment problem's constraint matrix is  unimodular and all right-hand-side values are integers. As a result, every extreme point of the LP relaxation is integral, so solving the linear program automatically yields binary assignment decisions without explicitly imposing  $ x_{ij} \in {0,1}$
